# Viewer 3.1 — presentation exports and NAV controls

This notebook derives `viewer/mouse_lung_v3.1.html` from the committed 3.0 viewer. It changes presentation controls and documentation, **not** the deposited M07 or E02 geometry. M07 coordinates remain local to that model. Ruberte et al. (2025) supports standardized quadruped anatomical **terminology**, not M07 dimensions or Xenium-to-M07 registration.

GIF recording samples the WebGL view at about 12 frames/s for at most 6 seconds; quantization and encoding run in the visitor's browser. PNG preserves full alpha. GIF has one-bit transparency. Both outputs omit the left control panel and retain a small orientation compass and, when shown, concise plane settings.

The GIF encoder is inlined into the generated HTML because local `file://` pages can block imports of adjacent module files. Its vendored source and license remain under `assets/`.

Version 3.1 now presents three explicitly bounded GIF capture profiles before recording. Width and frame rate increase visual detail but also raise client-side memory and encoding cost; none changes M07 geometry.

The M07 lobe layer starts at 95% opacity on each fresh load; the range control and material opacity are synchronized by dispatching the normal input event after the meshes exist.

The GIF has no fixed recording duration. Its encoder receives frames as they are captured, so raw frames do not accumulate in memory. The GIF byte stream still grows with recording length. When clipping is active at the start, the output uses a 16:9 scientific slide layout with the 3D view, actual 2D mesh section, NAV orientation and exact M07-local plane address.

In [1]:
from pathlib import Path
import hashlib, re

ROOT = Path.cwd()
assert (ROOT / 'viewer/mouse_lung_v3.0.html').exists(), 'Run from the repository root'
src = (ROOT / 'viewer/mouse_lung_v3.0.html').read_text()
def replace_once(old, new):
    global src
    count = src.count(old)
    assert count == 1, (old[:80], count)
    src = src.replace(old, new, 1)

print('Source bytes:', len(src.encode()), 'SHA256:', hashlib.sha256(src.encode()).hexdigest())

Source bytes: 44874700 SHA256: fc602ada64d9de2ccb23c135fc41e30e93c139a8ac6de944fa59c74335ae76ea


## 1. Presentation toolbar and compact plane controls
The icon buttons retain accessible names and keyboard focus. Exact numeric inputs supplement native sliders; the 2D drag pad changes azimuth and elevation together.

In [2]:
replace_once('<title>Mouse Lung Explorer 3.0 | anatomical reference</title>', '<title>Mouse Lung Explorer 3.1 | M07 model · NAV anatomy</title>')
replace_once('3.0 · C57BL/6 m07 anatomical reference', '3.1 · M07 model · NAV / Ruberte 2025')
replace_once('Mouse lung explorer · 3.0', 'Mouse lung explorer · 3.1')
replace_once('<output id="opacityOut">100%</output></label><input id="opacity" type="range" min="15" max="100" value="100"', '<output id="opacityOut">95%</output></label><input id="opacity" type="range" min="15" max="100" value="95"')
replace_once("  document.getElementById('airwayVisible').addEventListener('change'", "  opacity.dispatchEvent(new Event('input'));\n  document.getElementById('airwayVisible').addEventListener('change'")
replace_once("for(const id of ['depth','angle','angle2'])cutBody.appendChild(document.getElementById(id).closest('.control'));", "cutBody.appendChild(ui.querySelector('.planeControls'));")
replace_once("address.querySelector('.small').textContent='M07 frame: right, ventral, cranial. az = horizontal angle; el = elevation; d = signed distance from the model origin.';", "address.querySelector('.small').textContent='M07 local model frame (mm): right, ventral, cranial per NAV terminology discussed by Ruberte et al. (2025). az and el are plane angles; d is signed distance from the M07 origin. These are not Xenium coordinates.';")
replace_once('<button id="saveLungPng" type="button" class="captureButton" aria-label="Save lung view as PNG">Save PNG</button>\n  <button id="resumeRotateBtn" type="button">Resume rotation</button>', """<div id="presentationToolbar" role="toolbar" aria-label="Presentation export and motion">
    <button id="saveLungPng" type="button" class="iconButton" aria-label="Download transparent PNG" title="Download transparent PNG"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M4 7h3l1.5-2h7L17 7h3v12H4z"/><circle cx="12" cy="13" r="3.2"/></svg></button>
    <button id="recordLungGif" type="button" class="iconButton" aria-label="Choose GIF recording quality" title="Choose GIF recording quality"><svg viewBox="0 0 24 24" aria-hidden="true"><rect x="3" y="5" width="18" height="14" rx="2"/><circle cx="12" cy="12" r="3.1" fill="currentColor" stroke="none"/></svg></button><span id="gifTimer" hidden aria-hidden="true">00:00</span>
    <button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Resume automatic rotation" title="Resume or pause automatic rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20 8a8 8 0 1 0 1 6M20 3v5h-5"/></svg></button>
    <button id="rotationSpeed" type="button" class="speedButton" aria-label="Rotation speed 1 times" title="Change automatic rotation speed">1×</button>
  </div>
  <div id="gifSetup" class="panel" role="dialog" aria-label="GIF recording settings" hidden>
    <strong>Record GIF</strong><button id="gifCancel" type="button" aria-label="Close GIF settings" title="Close">×</button>
    <label for="gifQuality">Quality</label>
    <select id="gifQuality" aria-label="GIF capture quality">
      <option value="compact">Compact · 640 px · 12 fps</option>
      <option value="clear">Clear · 960 px · 15 fps</option>
      <option value="slide" selected>Slide · 1280 px · 20 fps</option>
    </select>
    <p id="gifEstimate" class="small" aria-live="polite"></p>
    <button id="gifStart" type="button" class="primary">Start recording</button>
  </div>""")
replace_once('<div class="control"><label for="depth"><span>Plane offset along its normal</span><output id="depthOut">0.00</output></label><input id="depth" type="range" min="-10" max="10" value="0" step="0.05" /></div>\n    <div class="control"><label for="angle"><span>Plane azimuth · LR ↔ DV</span><output id="angleOut">15°</output></label><input id="angle" type="range" min="-90" max="90" value="15" step="1" /></div>\n    <div class="control"><label for="angle2"><span>Plane elevation · Cr ↔ Cd</span><output id="angle2Out">0°</output></label><input id="angle2" type="range" min="-90" max="90" value="0" step="1" /></div>', """<div class="planeControls" aria-label="NAV-oriented model plane controls">
      <div class="planeControlHead"><strong>Section plane</strong><span>M07 frame · mm</span></div>
      <div class="planeControlRow"><label for="depth">Offset <small>normal distance</small></label><input id="depth" type="range" min="-10" max="10" value="0" step="0.05"/><input id="depthNumber" type="number" min="-10" max="10" step="0.05" value="0" aria-label="Plane offset in millimeters"/><span class="unit">mm</span><output id="depthOut" class="srOnly">0.00</output></div>
      <div class="planeControlRow"><label for="angle">Azimuth <small>right ↔ ventral</small></label><input id="angle" type="range" min="-90" max="90" value="15" step="1"/><input id="angleNumber" type="number" min="-90" max="90" step="1" value="15" aria-label="Azimuth in degrees"/><span class="unit">°</span><output id="angleOut" class="srOnly">15°</output></div>
      <div class="planeControlRow"><label for="angle2">Elevation <small>cranial ↔ caudal</small></label><input id="angle2" type="range" min="-90" max="90" value="0" step="1"/><input id="angle2Number" type="number" min="-90" max="90" step="1" value="0" aria-label="Elevation in degrees"/><span class="unit">°</span><output id="angle2Out" class="srOnly">0°</output></div>
      <div class="tiltPadWrap"><span>Drag to tilt</span><div id="tiltPad" role="application" aria-label="Drag to set azimuth and elevation; keyboard use numeric fields"><i id="tiltHandle"></i></div><span class="padHint">azimuth → · elevation ↑</span></div>
    </div>""")
replace_once("alveolarButton.textContent=\'Alveolar ROI · 3D + section\'", "alveolarButton.textContent='Alveolar chicken-wire view';alveolarButton.title='Explore an independent measured alveolar-region ROI (not registered to M07)'")
replace_once('Reference lung frame: +X right, +Y ventral, +Z cranial; mm. Plane n·x = d. Origin belongs to this exact mesh, not the body or Xenium slide.', 'M07 local frame (mm): +X right, +Y ventral, +Z cranial; plane n·x = d. Direction names follow NAV terminology discussed by Ruberte et al. (2025). The origin and dimensions belong to this M07 mesh, not to Ruberte et al. or Xenium.')
replace_once("const renderer=new THREE.WebGLRenderer({antialias:true,alpha:false,stencil:true,preserveDrawingBuffer:true,powerPreference:'high-performance'});", "const renderer=new THREE.WebGLRenderer({antialias:true,alpha:true,stencil:true,preserveDrawingBuffer:true,powerPreference:'high-performance'});")
replace_once("controls.autoRotateSpeed=1.00; let currentView", "controls.autoRotateSpeed=1.00; let currentView")
replace_once("function setResumeButton(){resumeRotateBtn.style.display=autoRotateStoppedByUser?'block':'none';}", "function setResumeButton(){resumeRotateBtn.setAttribute('aria-pressed',String(!autoRotateStoppedByUser));resumeRotateBtn.title=autoRotateStoppedByUser?'Resume automatic rotation':'Pause automatic rotation';}")
replace_once("resumeRotateBtn.addEventListener('click',()=>resumeAutoRotate());", "resumeRotateBtn.addEventListener('click',()=>autoRotateStoppedByUser?resumeAutoRotate():stopAutoRotateByUser());\n  const speedButton=document.getElementById('rotationSpeed');const speedValues=[1,1.5,2];let speedIndex=0;speedButton.addEventListener('click',()=>{speedIndex=(speedIndex+1)%speedValues.length;controls.autoRotateSpeed=speedValues[speedIndex];speedButton.textContent=speedValues[speedIndex]+'×';speedButton.setAttribute('aria-label','Rotation speed '+speedValues[speedIndex]+' times');});setResumeButton();")
print('Toolbar and plane controls patched')

Toolbar and plane controls patched


In [3]:
style = r"""
/* v3.1: deliberately quiet presentation controls */
#presentationToolbar{position:absolute;right:16px;top:16px;z-index:45;display:flex;gap:6px;padding:5px;border:1px solid rgba(124,216,255,.24);border-radius:16px;background:rgba(7,16,24,.85);backdrop-filter:blur(14px)}
#presentationToolbar button{position:static;display:grid;place-items:center;min-width:36px;width:36px;height:36px;min-height:36px;padding:6px;border-radius:11px;background:transparent;border:1px solid transparent;color:#dbf8ff;box-shadow:none}
#presentationToolbar button:hover,#presentationToolbar button[aria-pressed="true"]{border-color:rgba(124,216,255,.45);background:rgba(124,216,255,.13)}
#presentationToolbar button:focus-visible,.planeControlRow input:focus-visible,#tiltPad:focus-visible{outline:2px solid #83d7ff;outline-offset:2px}
#presentationToolbar svg{width:19px;height:19px;fill:none;stroke:currentColor;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round}
#presentationToolbar .speedButton{font-size:11px;font-weight:750;letter-spacing:-.02em}
#presentationToolbar #gifTimer{align-self:center;min-width:36px;color:#ff9999;font:700 10px ui-monospace,SFMono-Regular,monospace}#presentationToolbar #gifTimer[hidden]{display:none}
#presentationToolbar #recordLungGif[aria-pressed="true"]{color:#ff7474;border-color:#ff7474;background:rgba(255,85,85,.17)}
#gifSetup{position:absolute;right:16px;top:72px;z-index:47;width:min(294px,calc(100vw - 32px));padding:14px;background:rgba(8,17,25,.96);border-color:rgba(124,216,255,.35)}
#gifSetup[hidden]{display:none}#gifSetup strong{font-size:12px}#gifSetup #gifCancel{float:right;min-width:26px;width:26px;height:26px;min-height:26px;padding:0;font-size:19px;line-height:1}
#gifSetup label{display:block;margin-top:12px;margin-bottom:5px;color:var(--muted);font-size:10px}#gifQuality{width:100%;padding:8px;color:var(--text);background:#12222e;border:1px solid rgba(124,216,255,.35);border-radius:8px;font:11px system-ui,sans-serif}
#gifEstimate{min-height:43px;margin:9px 0;color:var(--muted)}#gifStart{width:100%}#sideHint{top:77px}
.srOnly{position:absolute!important;width:1px;height:1px;padding:0;margin:-1px;overflow:hidden;clip:rect(0,0,0,0);white-space:nowrap;border:0}
.planeControls{margin:12px 0 10px;padding:11px;border:1px solid rgba(124,216,255,.2);background:rgba(7,18,28,.62);border-radius:14px}
.planeControlHead{display:flex;align-items:baseline;justify-content:space-between;margin-bottom:7px;font-size:11px}.planeControlHead span{font-size:9px;color:var(--muted)}
.planeControlRow{display:grid;grid-template-columns:85px minmax(45px,1fr) 54px 16px;gap:7px;align-items:center;min-height:43px;border-top:1px solid rgba(255,255,255,.06)}
.planeControlRow label{font-size:10.5px;color:#e7f6fc}.planeControlRow small{display:block;font-size:8.5px;line-height:1.2;color:var(--muted)}
.planeControlRow input[type="range"]{width:100%;accent-color:#80d9fc;margin:0;cursor:ew-resize}
.planeControlRow input[type="number"]{width:54px;height:27px;padding:2px 3px;color:#eafaff;background:rgba(124,216,255,.07);border:1px solid rgba(124,216,255,.26);border-radius:7px;text-align:right;font:10px ui-monospace,SFMono-Regular,monospace;appearance:textfield}
.planeControlRow input[type="number"]::-webkit-inner-spin-button{-webkit-appearance:none;margin:0}.planeControlRow .unit{color:#96a9b8;font-size:10px}
.tiltPadWrap{display:flex;align-items:center;gap:8px;margin-top:8px;color:var(--muted);font-size:9px}.tiltPadWrap>span:first-child{min-width:69px}
#tiltPad{position:relative;width:72px;height:72px;flex:none;border-radius:10px;border:1px solid rgba(124,216,255,.28);background:linear-gradient(90deg,transparent calc(50% - .5px),rgba(124,216,255,.16) 50%,transparent calc(50% + .5px)),linear-gradient(transparent calc(50% - .5px),rgba(124,216,255,.16) 50%,transparent calc(50% + .5px)),rgba(124,216,255,.035);touch-action:none;cursor:crosshair}
#tiltHandle{position:absolute;width:11px;height:11px;border-radius:50%;background:#83d7ff;box-shadow:0 0 0 4px rgba(124,216,255,.15);transform:translate(-50%,-50%);left:50%;top:50%;pointer-events:none}.padHint{line-height:1.35}
@media(max-width:700px){#presentationToolbar{right:10px;top:10px;gap:2px}#presentationToolbar button{width:32px;height:32px;min-width:32px}#sideHint{top:63px}}
"""
replace_once('</style>', style+'</style>')

numeric_js = r"""
  // Exact values remain synchronized after slider, pad, link-restore or preset changes.
  const planeNumbers=[['depth','depthNumber',2],['angle','angleNumber',1],['angle2','angle2Number',1]];
  for(const [sliderId,numberId,precision] of planeNumbers){
    const slider=document.getElementById(sliderId),number=document.getElementById(numberId);
    slider.addEventListener('input',()=>{number.value=Number(slider.value).toFixed(precision)});
    number.addEventListener('input',()=>{const value=Math.max(Number(slider.min),Math.min(Number(slider.max),Number(number.value)));if(Number.isFinite(value)){slider.value=value;slider.dispatchEvent(new Event('input',{bubbles:true}))}});
  }
  const tiltPad=document.getElementById('tiltPad'),tiltHandle=document.getElementById('tiltHandle');
  function syncPlaneControls(){for(const [sliderId,numberId,precision] of planeNumbers){const slider=document.getElementById(sliderId),number=document.getElementById(numberId);if(document.activeElement!==number)number.value=Number(slider.value).toFixed(precision)}tiltHandle.style.left=((Number(angle.value)+90)/180*100)+'%';tiltHandle.style.top=((90-Number(angle2.value))/180*100)+'%'}
  function moveTilt(event){const r=tiltPad.getBoundingClientRect(),x=Math.max(0,Math.min(1,(event.clientX-r.left)/r.width)),y=Math.max(0,Math.min(1,(event.clientY-r.top)/r.height));angle.value=Math.round(x*180-90);angle2.value=Math.round(90-y*180);angle.dispatchEvent(new Event('input',{bubbles:true}));angle2.dispatchEvent(new Event('input',{bubbles:true}));syncPlaneControls()}
  tiltPad.addEventListener('pointerdown',event=>{tiltPad.setPointerCapture(event.pointerId);moveTilt(event)});
  tiltPad.addEventListener('pointermove',event=>{if(tiltPad.hasPointerCapture(event.pointerId))moveTilt(event)});
  syncPlaneControls();
"""
replace_once('  // v26 visibility aid and unregistered source-section panel.', numeric_js+'\n  // v26 visibility aid and unregistered source-section panel.')
print('CSS and fine plane interactions patched')

CSS and fine plane interactions patched


## 2. Transparent image and browser-side GIF
The WebGL scene is rendered with an alpha channel only for capture, then restored. The orientation overlay is drawn from the live NAV compass geometry; browser panels are never copied into the export. GIF is palette-quantized with a single transparent index, so soft alpha edges cannot survive.

In [4]:
capture_js = r"""
  // Export the geometry canvas, not a screenshot of page controls.
  const capturePngButton=document.getElementById('saveLungPng');
  const gifButton=document.getElementById('recordLungGif');
  function drawCompactCompass(ctx,w,h,onLeft=false){
    const svg=document.getElementById('orientationSvg'),size=Math.max(66,Math.min(100,w*.13)),x=onLeft?18:w-size-18,y=h-size-22,scale=size/128;
    ctx.save();ctx.translate(x,y);ctx.scale(scale,scale);
    for(const line of svg.querySelectorAll('line')){ctx.beginPath();ctx.moveTo(Number(line.getAttribute('x1')),Number(line.getAttribute('y1')));ctx.lineTo(Number(line.getAttribute('x2')),Number(line.getAttribute('y2')));ctx.strokeStyle=line.getAttribute('stroke')||'#d7f3ff';ctx.globalAlpha=Number(line.getAttribute('opacity')||1);ctx.lineWidth=2.5;ctx.lineCap='round';ctx.stroke()}
    ctx.globalAlpha=1;ctx.fillStyle='#d7f3ff';ctx.beginPath();ctx.arc(64,64,3,0,Math.PI*2);ctx.fill();ctx.font='bold 10px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='middle';
    for(const label of svg.querySelectorAll('text')){ctx.fillStyle=label.getAttribute('fill')||'#d7f3ff';ctx.fillText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')))}ctx.restore();
    ctx.save();ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061019';ctx.shadowBlur=5;ctx.font='600 '+Math.max(9,Math.round(w/110))+'px system-ui,sans-serif';ctx.textAlign='left';ctx.fillText('ANATOMICAL ORIENTATION',x,h-size-29);ctx.restore();
  }
  function presentationFrame(maxWidth,forGif=false,showSection=false){
    const source=renderer.domElement,ratio=forGif?maxWidth/source.width:Math.min(1,maxWidth/source.width),w=Math.max(2,Math.round(source.width*ratio));
    const oldBackground=scene.background;scene.background=null;renderer.setClearColor(0x000000,0);renderer.render(scene,camera);
    const h=forGif&&showSection?Math.round(w*9/16):Math.max(2,Math.round(source.height*ratio));
    const out=document.createElement('canvas');out.width=w;out.height=h;const ctx=out.getContext('2d',{willReadFrequently:true});ctx.clearRect(0,0,w,h);
    if(forGif&&showSection){
      const gap=Math.max(10,Math.round(w*.014)),panelX=Math.round(w*.64),leftW=panelX-gap,boxX=panelX,boxW=w-boxX-gap;
      const cropRatio=leftW/h;let cropH=source.height,cropW=cropH*cropRatio;
      if(cropW>source.width){cropW=source.width;cropH=cropW/cropRatio}
      ctx.drawImage(source,(source.width-cropW)/2,(source.height-cropH)/2,cropW,cropH,0,0,leftW,h);
      ctx.fillStyle='rgba(9,22,31,.94)';ctx.fillRect(boxX,gap,boxW,h-2*gap);ctx.strokeStyle='rgba(145,218,237,.45)';ctx.lineWidth=1;ctx.strokeRect(boxX+.5,gap+.5,boxW-1,h-2*gap-1);
      const pad=Math.max(12,Math.round(w*.013)),textX=boxX+pad,innerW=boxW-2*pad;
      ctx.fillStyle='#9ee9f4';ctx.font='700 '+Math.max(10,Math.round(w/95))+'px system-ui,sans-serif';ctx.fillText('2D SECTION · M07 MODEL',textX,gap+pad+3);
      ctx.fillStyle='#d9eaf2';ctx.font=Math.max(9,Math.round(w/115))+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+21,innerW);
      const addressY=h-gap-Math.max(94,Math.round(h*.17));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+34,imageH=Math.max(20,Math.min(innerW*section.height/section.width,addressY-imageTop-12)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(section,textX+(innerW-imageW)/2,imageTop,imageW,imageH);
      ctx.strokeStyle='rgba(145,218,237,.24)';ctx.beginPath();ctx.moveTo(textX,addressY-12);ctx.lineTo(textX+innerW,addressY-12);ctx.stroke();
      ctx.fillStyle='#9ee9f4';ctx.font='700 '+Math.max(9,Math.round(w/116))+'px system-ui,sans-serif';ctx.fillText('SECTION PLANE ADDRESS · M07 LOCAL',textX,addressY+3);
      const az=sectionAngle*Math.PI/180,el=sectionTilt*Math.PI/180,nx=Math.cos(el)*Math.cos(az),ny=Math.cos(el)*Math.sin(az),nz=Math.sin(el);
      ctx.fillStyle='#d9eaf2';ctx.font=Math.max(9,Math.round(w/130))+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText('Frame +R / +V / +Cr · mm',textX,addressY+23,innerW);
      ctx.fillText('n = ('+[nx,ny,nz].map(v=>v.toFixed(3)).join(', ')+')',textX,addressY+41,innerW);
      ctx.fillText('n · x = '+sectionDepth.toFixed(3)+' mm',textX,addressY+59,innerW);
      drawCompactCompass(ctx,leftW,h,true);
    }else{
      ctx.drawImage(source,0,0,w,h);drawCompactCompass(ctx,w,h);
      if(sectionPlane.visible){ctx.save();ctx.font=Math.max(10,Math.round(w/75))+'px system-ui,sans-serif';ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;const line='M07 · az '+Number(angle.value).toFixed(0)+'° · el '+Number(angle2.value).toFixed(0)+'° · d '+Number(depth.value).toFixed(2)+' mm';ctx.fillText(line,Math.max(12,w*.02),h-Math.max(14,h*.04));ctx.restore()}
    }
    if(wholeAlveolarGroup.visible){ctx.save();ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;ctx.font=Math.max(9,Math.round(w/110))+'px system-ui,sans-serif';ctx.fillText('SCHEMATIC AIRSPACES · ILLUSTRATIVE',Math.max(12,w*.016),Math.max(18,h*.04));ctx.restore()}
    scene.background=oldBackground;renderer.render(scene,camera);return out;
  }
  capturePngButton.addEventListener('click',()=>saveCanvasPng(presentationFrame(2200),'mouse-lung-M07-v3.1-transparent.png'));
  const gifSetup=document.getElementById('gifSetup'),gifQuality=document.getElementById('gifQuality'),gifEstimate=document.getElementById('gifEstimate'),gifTimer=document.getElementById('gifTimer');
  const gifPresets={compact:{width:640,fps:12},clear:{width:960,fps:15},slide:{width:1280,fps:20}};
  let gifRecording=false,gifExporting=false,gifStart=0,gifLast=-Infinity,gifModule=null,gifOptions=gifPresets.slide,gifEncoder=null,gifFrameCount=0,gifLayoutClipped=false;
  function updateGifEstimate(){
    const option=gifPresets[gifQuality.value],w=option.width,h=clipping?Math.round(w*9/16):Math.round(w*renderer.domElement.height/renderer.domElement.width),megabytes=(w*h*4/1048576).toFixed(1);
    gifEstimate.textContent='No time limit · click the red record icon to stop. Frames are encoded as you record ('+megabytes+' MB raw per frame); long GIFs and higher quality use more local CPU and storage.';
  }
  gifQuality.addEventListener('change',updateGifEstimate);
  document.getElementById('gifCancel').addEventListener('click',()=>gifSetup.hidden=true);
  gifSetup.addEventListener('keydown',event=>{if(event.key==='Escape')gifSetup.hidden=true});
  function recordTick(now){
    if(!gifRecording)return;
    if(now-gifLast>=1000/gifOptions.fps){
      try{
        const frame=presentationFrame(gifOptions.width,true,gifLayoutClipped),pixels=frame.getContext('2d',{willReadFrequently:true}).getImageData(0,0,frame.width,frame.height).data;
        for(let p=0;p<pixels.length;p+=4)pixels[p+3]=pixels[p+3]<128?0:255;
        const palette=gifModule.quantize(pixels,256,{format:'rgba4444',oneBitAlpha:true,clearAlpha:true}),index=gifModule.applyPalette(pixels,palette,'rgba4444'),transparentIndex=palette.findIndex(color=>color[3]===0);
        const delay=gifFrameCount?Math.max(20,Math.min(1000,Math.round((now-gifLast)/10)*10)):Math.round(1000/gifOptions.fps/10)*10;
        gifEncoder.writeFrame(index,frame.width,frame.height,{palette,delay,transparent:transparentIndex>=0,transparentIndex:Math.max(0,transparentIndex),dispose:2});
        gifFrameCount++;gifLast=now;const elapsed=Math.floor((now-gifStart)/1000);gifTimer.textContent=String(Math.floor(elapsed/60)).padStart(2,'0')+':'+String(elapsed%60).padStart(2,'0');gifButton.title='Stop recording · '+gifTimer.textContent+' · '+gifFrameCount+' frames';
      }catch(error){console.error('GIF frame failed',error);gifRecording=false;gifEncoder=null;gifTimer.hidden=true;gifButton.setAttribute('aria-pressed','false');window.alert('GIF capture stopped: '+error.message);return}
    }
    requestAnimationFrame(recordTick);
  }
  function finishGif(){
    if(!gifRecording)return;gifRecording=false;gifExporting=true;gifTimer.hidden=true;gifButton.setAttribute('aria-pressed','false');gifButton.setAttribute('aria-label','Choose GIF recording quality');document.getElementById('toggleClip').disabled=false;
    if(!gifFrameCount){gifEncoder=null;gifExporting=false;gifButton.title='No frames recorded';return}
    try{gifButton.title='Finalizing GIF…';gifEncoder.finish();const blob=new Blob([gifEncoder.bytes()],{type:'image/gif'}),url=URL.createObjectURL(blob),link=document.createElement('a');link.href=url;link.download='mouse-lung-M07-v3.1-'+gifQuality.value+'.gif';document.body.appendChild(link);link.click();link.remove();setTimeout(()=>URL.revokeObjectURL(url),10000);gifButton.title='GIF ready · '+gifFrameCount+' frames · '+(blob.size/1048576).toFixed(1)+' MB'}catch(error){console.error('GIF export failed',error);gifButton.title='GIF export failed';window.alert('GIF export failed: '+error.message)}finally{gifEncoder=null;gifExporting=false;gifFrameCount=0}
  }
  gifButton.addEventListener('click',()=>{
    if(gifRecording){finishGif();return}
    if(gifExporting)return;
    gifSetup.hidden=false;updateGifEstimate();gifQuality.focus();
  });
  document.getElementById('gifStart').addEventListener('click',async()=>{
    if(gifRecording||gifExporting)return;
    gifOptions=gifPresets[gifQuality.value];gifSetup.hidden=true;gifButton.title='Loading embedded GIF encoder…';
    try{gifModule=gifModule||await import('../assets/gifenc.esm.js')}catch(error){console.error(error);gifButton.title='GIF encoder unavailable';window.alert('Could not load the embedded GIF encoder.');return}
    gifEncoder=gifModule.GIFEncoder();gifFrameCount=0;gifLayoutClipped=clipping;if(gifLayoutClipped)drawSlice();document.getElementById('toggleClip').disabled=true;
    gifRecording=true;gifStart=performance.now();gifLast=-Infinity;gifTimer.hidden=false;gifTimer.textContent='00:00';gifButton.setAttribute('aria-pressed','true');gifButton.setAttribute('aria-label','Stop GIF recording');requestAnimationFrame(recordTick);
  });
"""
old_capture = """  document.getElementById('saveLungPng').addEventListener('click',()=>{
    document.getElementById('saveLungPng').textContent='Preparing PNG…';
    renderer.render(scene,camera);
    if(clipping)drawSlice();
    const section=clipping?document.getElementById('sliceCanvas'):null;
    const schematic=wholeAlveolarGroup.visible;
    const label=schematic?'schematic airspaces enabled; illustrative size/count':'measured M07 gross geometry';
    compositeCapture(renderer.domElement,section,'M07 anatomical reference · version 3.0',label,'mouse-lung-v3.0.png');
  });"""
replace_once(old_capture, capture_js)
# Inline the small vendored encoder as a data-URL module: file:// viewers cannot fetch sibling ES modules reliably.
import base64
encoder = (ROOT / 'assets/gifenc.esm.js').read_bytes()
encoded = base64.b64encode(encoder).decode('ascii')
replace_once("import('../assets/gifenc.esm.js')", f"import('data:text/javascript;base64,{encoded}')")
replace_once("document.getElementById('saveLungPng').textContent='Save PNG';", '')
replace_once('controls.update();updateOrientationOverlay();updateSelectedLabel();', 'controls.update();updateOrientationOverlay();updateSelectedLabel();syncPlaneControls();')
print('Transparent PNG and GIF export patched')

Transparent PNG and GIF export patched


## 3. Versioned output and structural checks
The GIF encoder is vendored under `assets/` for the build and embedded as a data-URL ES module in the output HTML. This lets the viewer work when opened directly via `file://` and keeps GitHub Pages static. Interaction quality still requires a browser QA pass.

In [5]:
out = ROOT / 'viewer/mouse_lung_v3.1.html'
out.write_text(src)
assert 'alpha:true' in src and "import('data:text/javascript;base64," in src and 'gifEncoder.writeFrame' in src
assert 'Mouse lung explorer · 3.1' in src and 'Alveolar chicken-wire view' in src and 'gifPresets' in src and 'gifSetup' in src and 'SECTION PLANE ADDRESS' in src
assert '<output id="opacityOut">95%</output>' in src and 'opacity.dispatchEvent' in src
assert 'Ruberte et al. (2025)' in src and 'M07 local frame (mm)' in src
assert 'textContent=\'Preparing PNG…\'' not in src
print('Built:', out, 'bytes:', out.stat().st_size, 'SHA256:', hashlib.sha256(out.read_bytes()).hexdigest())

Built: /Users/ashi/github/mouse-lung-explorer/viewer/mouse_lung_v3.1.html bytes: 44905750 SHA256: 48989e1b0a2e53faaa576da7dace7b1350716069f5bb5a87ae2f0976fb1e839e
